# Benchmark
A comprehensive comparison against state-of-the-art causal discovery methodologies is performed. <br>
Each selected method exemplifies a distinct category within the spectrum of causal inference approaches. <br>
Namely, we includes the Pairwise Granger Causality test implementation available in the `statsmodels` package for Python <br>

From the constraint-based family, we select the PCMCI algorithm developed by Runge, which is implemented in the Tigramite Python package. <br> 
The VarLiNGAM method, as proposed by Hyvärinen et al., is our choice for the noise-based category.  <br> 
It is implemented in Python via the LiNGAM library .   <br> 

Finally, for the score-based category, we incorporate DYNOTEARS, a method introduced by Pamfil et al. and implemented in the CausalNex Python library.  <br> 

We also include standard VAR modeling, adopting the regression coefficients as if they were causal.  <br> 


It is important to note that each method has its own underlying assumptions, which might not always be respected in practical scenarios. For example, VarLiNGAM assumes non-Gaussian errors, which is not the case in our experiments. Similarly, we use PCMCI with the ParCorr independence test; while a nonlinear test would be more appropriate. We faced computational challenges with PCMCI when attempting to use its nonlinear inpendence test CMIknn. Nevertheless, our goal is not to demonstrate that our approach outperforms all others under all conditions. Instead, we aim to show that our method can be a valuable addition to the toolbox for causal discovery in time series, offering unique insights and potentially complementing existing techniques. The adopted conditions for each method might be suboptimal for the given dataset, yet they provide a robust benchmark to evaluate the relative strengths and potential applications of our proposed approach.

Each method has been wrapped conveniently to uniformize the way to create the objects, run the execution, and return the same structure.  

In [5]:
import os

# This is because VARLINGAM will use all available CPU with n_jobs > 1 - Limit to 1 thread
os.environ['MKL_NUM_THREADS'] = '1'  
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.append('../src')


import pickle 
import os
from d2c.descriptors import DataLoader
from d2c.benchmark import VARLiNGAM, PCMCI, Granger, DYNOTEARS, D2CWrapper, VAR, MultivariateGranger

from imblearn.ensemble import BalancedRandomForestClassifier

#suppress future warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import time



In [6]:
N_JOBS = 5
MAXLAGS = 3

In [7]:
dataloader = DataLoader(n_variables = 50,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/dream3/dream3_50.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

Each method from the benchmark will take as input 
- `ts_list`: a list of `np.arrays` containing the values of the time series, 
- `maxlags`: the maxlags, 
- `n_jobs`: the number of jobs. <br>
It's important to notice that `get_causal_dfs()` will return a dictionary of dataframes where the key is the index of the corresponding time series from the input list `ts_list`.
So, if our data contains `15` time series and you want to access the last one we can do `causal_dfs[15 - 1]`

## Competitors

In [8]:
start_time_var = time.perf_counter()
var = VAR(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
var.run()
causal_dfs_var = var.get_causal_dfs()
end_time_var = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

In [9]:
start_time_varlingam = time.perf_counter()
varlingam = VARLiNGAM(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
varlingam.run()
causal_dfs_varlingam = varlingam.get_causal_dfs()
end_time_varlingam = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
start_time_pcmci = time.perf_counter()
pcmci = PCMCI(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
pcmci.run()
causal_dfs_pcmci = pcmci.get_causal_dfs()
end_time_pcmci = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# pcmci_gdpc = PCMCI(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS, ci ='GPDC')
# pcmci_gdpc.run()
# causal_dfs_pcmci_gdpc = pcmci_gdpc.get_causal_dfs()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/miniconda3/

In [11]:
start_time_granger = time.perf_counter()
granger = Granger(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
granger.run()
causal_dfs_granger = granger.get_causal_dfs()
end_time_granger = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
start_time_dynotears = time.perf_counter()
dynotears = DYNOTEARS(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
dynotears.run()
causal_dfs_dynotears = dynotears.get_causal_dfs()
end_time_dynotears = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

In [13]:
start_time_mvgc = time.perf_counter()
mvgc = MultivariateGranger(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
mvgc.run()
causal_dfs_mvgc = mvgc.get_causal_dfs()
end_time_mvgc = time.perf_counter()

Running parallel inference over 5 time series using 5 jobs...


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# save all in a temp folder pickle 
temp_folder = 'data/benchmark_results'
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)


In [ ]:

all = {
    'causal_dfs_var': causal_dfs_var,
    'causal_dfs_varlingam': causal_dfs_varlingam,
    'causal_dfs_pcmci': causal_dfs_pcmci,
    'causal_dfs_pcmci_gpdc': None, 
    'causal_dfs_granger': causal_dfs_granger,
    'causal_dfs_mvgc': causal_dfs_mvgc,
    'causal_dfs_dynotears': causal_dfs_dynotears,
    'observations': original_observations_testing,
    'dags': flattened_dags_testing,
    'true_causal_dfs': true_causal_dfs
}
pickle.dump(all, open(os.path.join(temp_folder, 'causal_dfs_before_d2c_dream3.pkl'), 'wb'))

In [ ]:
# # load everything
# all = pickle.load(open(os.path.join(temp_folder, 'causal_dfs_before_d2c.pkl'), 'rb'))
# causal_dfs_var = all['causal_dfs_var']
# causal_dfs_varlingam = all['causal_dfs_varlingam']
# causal_dfs_pcmci = all['causal_dfs_pcmci']
# causal_dfs_granger = all['causal_dfs_granger']
# causal_dfs_dynotears = all['causal_dfs_dynotears']

## D2CWrapper
For coherence with the other results, a D2CWrapper class has been created that behave exactly like the other approaches.<br>
It therefore exposes the methods `run()` and `get_causal_dfs()`. <br>
It requires a model that has been trained already and it will compute descriptors for unseen data of which the DAG is ignored. <br>
In this case, the model cannot select a subset of features (no `couples_to_consider_per_dag` attribute).
The predictions from the model on the newly computed descriptors are the labels that will be provided in the causal df. <br>

<b>Important:</b> make sure your model has been trained on the same feature set. If you have used `full=True` when generating the training descriptors, you should use `full=True` here as well

In [14]:
import pandas as pd
descriptors_df_train = pd.read_pickle('data/descriptors_df_train.pkl')

X_train = descriptors_df_train.drop(columns=['graph_id','edge_source','edge_dest','is_causal'])
y_train = descriptors_df_train['is_causal']

clf = BalancedRandomForestClassifier(n_estimators=500, max_depth=None, random_state=0, sampling_strategy='auto',replacement=True,bootstrap=True)
clf.fit(X_train, y_train)

BalancedRandomForestClassifier(bootstrap=True, n_estimators=500, random_state=0,
                               replacement=True, sampling_strategy='auto')

In [21]:
d2cwrapper = D2CWrapper(
    ts_list=original_observations_testing,
    model=clf,
    n_variables=50,
    maxlags=MAXLAGS,
    mb_estimator = 'ts',
    n_jobs=60, 
    full=True,
    dynamic=True,
    manages_own_parallelism=True,
)

start_time_d2c = time.perf_counter()
d2cwrapper.run()
causal_dfs_d2c = d2cwrapper.get_causal_dfs()
end_time_d2c = time.perf_counter()

Running in single-threaded mode or managing parallelism manually.


Processing Time Series:   0%|          | 0/5 [00:00<?, ?it/s]

Running descriptor computation in parallel with 60 jobs...


Computing Descriptors:   0%|          | 0/7500 [00:00<?, ?it/s]

Running descriptor computation in parallel with 60 jobs...


...

Computing Descriptors:   0%|          | 0/7500 [00:00<?, ?it/s]

Running descriptor computation in parallel with 60 jobs...


Computing Descriptors:   0%|          | 0/7500 [00:00<?, ?it/s]

Running descriptor computation in parallel with 60 jobs...


Computing Descriptors:   0%|          | 0/7500 [00:00<?, ?it/s]

## Saving

In [22]:
with open('data/causal_dfs_dream3.pkl', 'wb') as f:
    pickle.dump((causal_dfs_var, 
                None, # causal_dfs_varlingam, 
                causal_dfs_pcmci,
                None, # causal_dfs_mvgc,
                None, # causal_dfs_pcmci_gdpc,
                causal_dfs_granger, 
                causal_dfs_dynotears,
                causal_dfs_d2c, 
                true_causal_dfs), f)

In [20]:
causal_dfs_d2c

{0:      from  to effect  p_value  probability  is_causal
 0      10   0   None    0.258        0.742          1
 1      10   1   None    0.538        0.462          0
 2      10   2   None    0.768        0.232          0
 3      10   3   None    0.144        0.856          1
 4      10   4   None    0.500        0.500          0
 ..    ...  ..    ...      ...          ...        ...
 295    39   5   None    0.666        0.334          0
 296    39   6   None    0.668        0.332          0
 297    39   7   None    0.630        0.370          0
 298    39   8   None    0.842        0.158          0
 299    39   9   None    0.458        0.542          1
 
 [300 rows x 6 columns],
 1:      from  to effect  p_value  probability  is_causal
 0      10   0   None    0.248        0.752          1
 1      10   1   None    0.620        0.380          0
 2      10   2   None    0.716        0.284          0
 3      10   3   None    0.718        0.282          0
 4      10   4   None    0.664  